# ESG/SDG News Sentiment Portfolio Construction — NIFTY 50

An end-to-end pipeline that turns company news into ESG/SDG sentiment signals, combines them with price features, and backtests ML-driven stock-selection portfolios on the NIFTY 50.

**Inspired by:** Feng et al., "Sustainable Portfolio Construction via Machine Learning: ESG, SDG and Sentiment", *European Financial Management* (2025), 31:1148–1169, DOI: [10.1111/eufm.12531](https://doi.org/10.1111/eufm.12531)

**What this notebook does:**
1. Collects ESG-related headlines for the 50 NIFTY 50 companies (GDELT, with a Google News RSS fallback)
2. Scores each headline with VADER sentiment and ESG / SDG / controversy keyword lexicons
3. Aggregates to daily company-level scores and merges them with daily returns from yfinance
4. Trains four regressors (Linear, LASSO, Random Forest, Gradient Boosting) to predict next-day returns, using a chronological 70/30 split
5. Backtests monthly top-10 equal-weight portfolios chosen by predicted return, against an equal-weight benchmark of all stocks
6. Reports annualized return, volatility and Sharpe ratio, and saves the charts

**Engineering notes:** vectorized pandas/NumPy scoring, pre-compiled regex lexicons, and cached intermediate files so each stage can be re-run on its own.


## 0. Install Dependencies


Install everything from the repository root before running the notebook:

```bash
pip install -r requirements.txt
```


## 1. Configuration (Paper-Aligned)

In [2]:
"""
config — Central configuration aligned with Feng et al. (2024) methodology.
"""
import os, warnings
from datetime import datetime, timedelta
from pathlib import Path
import numpy as np

warnings.filterwarnings("ignore")

# ── Paths ──────────────────────────────────────────────────────────────
ROOT_DIR = Path(".").resolve()
PATHS = {
    "root":            ROOT_DIR,
    "data_raw":        ROOT_DIR / "data" / "raw",
    "data_processed":  ROOT_DIR / "data" / "processed",
    "outputs_models":  ROOT_DIR / "outputs" / "models",
    "outputs_figures": ROOT_DIR / "outputs" / "figures",
    "outputs_tables":  ROOT_DIR / "outputs" / "tables",
    "logs":            ROOT_DIR / "logs",
}
for p in PATHS.values():
    p.mkdir(parents=True, exist_ok=True)

FILES = {
    "companies":         PATHS["data_raw"]       / "nifty50_companies.csv",
    "raw_news":          PATHS["data_raw"]       / "raw_news.csv",
    "cleaned_news":      PATHS["data_processed"] / "cleaned_news.csv",
    "scored_news":       PATHS["data_processed"] / "scored_news.csv",
    "daily_scores":      PATHS["data_processed"] / "daily_scores.csv",
    "stock_returns":     PATHS["data_processed"] / "stock_returns.csv",
    "model_dataset":     PATHS["data_processed"] / "model_dataset.csv",
    "model_performance": PATHS["outputs_models"] / "model_performance.csv",
    "feature_importance":PATHS["outputs_models"] / "feature_importance.csv",
    "portfolio_perf":    PATHS["outputs_tables"] / "portfolio_performance.csv",
    "portfolio_returns": PATHS["outputs_tables"] / "portfolio_returns.csv",
}

# ── Date Range (pinned so results are reproducible; paper uses Feb 2015 – Feb 2022) ──
START_DATE = datetime(2025, 5, 7)
END_DATE   = datetime(2026, 5, 7)   # 1 year
YFINANCE_START = START_DATE.strftime("%Y-%m-%d")
YFINANCE_END   = END_DATE.strftime("%Y-%m-%d")

# ── GDELT API ──────────────────────────────────────────────────────────
GDELT = {
    "base_url":    "https://api.gdeltproject.org/api/v2/doc/doc",
    "max_records": 250,
    "chunk_days":  30,
    "sleep_sec":   0.3,
    "retries":     3,
}

# ── ESG Keywords ───────────────────────────────────────────────────────
ESG_KEYWORDS = [
    "ESG", "sustainability", "sustainable development", "SDG",
    "climate change", "carbon emissions", "net zero", "renewable energy",
    "corporate governance", "CSR", "greenwashing", "diversity", "human rights",
]

KW_ENVIRONMENTAL = [
    "climate", "carbon", "emission", "renewable", "solar", "wind", "green",
    "pollution", "deforestation", "biodiversity", "net zero", "fossil",
    "energy transition", "greenhouse", "environmental", "sustainability",
    "clean energy", "carbon neutral", "circular economy", "waste", "recycl",
]
KW_SOCIAL = [
    "labour", "labor", "worker", "employee", "human rights", "diversity",
    "inclusion", "gender", "equality", "community", "health", "safety",
    "supply chain", "social", "CSR", "welfare", "education",
]
KW_GOVERNANCE = [
    "governance", "board", "director", "audit", "compliance", "transparency",
    "accountability", "corruption", "shareholder", "executive pay", "SEBI",
    "disclosure", "regulatory", "insider trading", "ESG score",
]
KW_SDG = [
    "SDG", "sustainable development goal", "no poverty", "zero hunger",
    "gender equality", "clean water", "affordable energy", "decent work",
    "climate action", "responsible consumption", "UN agenda 2030",
]
KW_CONTROVERSY = [
    "fraud", "scandal", "penalty", "fine", "lawsuit", "violation",
    "corruption", "bribe", "greenwashing", "strike", "protest", "toxic",
]

# ── Paper-Aligned ML Settings ──────────────────────────────────────────
ML_CONFIG = {
    "rolling_windows": [63, 126],    # ~3 months and ~6 months in trading days
    "top_m_stocks":    [10, 30],     # Paper: M = 10 and 30
    "score_types":     ["esg", "sdg", "overall"],
    "cv_folds":        5,
    "random_state":    42,
}

# ── Portfolio Settings ─────────────────────────────────────────────────
PORTFOLIO = {
    "risk_free_rate": 0.065,   # 6.5% annualized (India)
    "trading_days":   252,
    "transaction_cost_bp": 1,  # 1 basis point per trade (paper default)
}

print("Configuration loaded.")
print(f"Date range: {YFINANCE_START} to {YFINANCE_END}")


Configuration loaded.
Date range: 2025-05-07 to 2026-05-07


## 2. NIFTY 50 Company Master

In [3]:
import pandas as pd

NIFTY50 = [
    ("Reliance Industries",        "RELIANCE",    "RELIANCE.NS",    "Reliance Industries|Reliance|RIL"),
    ("TCS",                        "TCS",         "TCS.NS",         "Tata Consultancy Services|TCS"),
    ("HDFC Bank",                  "HDFCBANK",    "HDFCBANK.NS",    "HDFC Bank"),
    ("ICICI Bank",                 "ICICIBANK",   "ICICIBANK.NS",   "ICICI Bank|ICICI"),
    ("Infosys",                    "INFY",        "INFY.NS",        "Infosys|INFY"),
    ("Hindustan Unilever",         "HINDUNILVR",  "HINDUNILVR.NS",  "Hindustan Unilever|HUL"),
    ("ITC",                        "ITC",         "ITC.NS",         "ITC Limited|ITC"),
    ("SBI",                        "SBIN",        "SBIN.NS",        "State Bank of India|SBI"),
    ("Bajaj Finance",              "BAJFINANCE",  "BAJFINANCE.NS",  "Bajaj Finance"),
    ("Bharti Airtel",              "BHARTIARTL",  "BHARTIARTL.NS",  "Bharti Airtel|Airtel"),
    ("Kotak Mahindra Bank",        "KOTAKBANK",   "KOTAKBANK.NS",   "Kotak Mahindra|Kotak Bank"),
    ("Wipro",                      "WIPRO",       "WIPRO.NS",       "Wipro"),
    ("L&T",                        "LT",          "LT.NS",          "Larsen & Toubro|L&T|LT"),
    ("Asian Paints",               "ASIANPAINT",  "ASIANPAINT.NS",  "Asian Paints"),
    ("HCL Tech",                   "HCLTECH",     "HCLTECH.NS",     "HCL Technologies|HCL Tech"),
    ("Maruti Suzuki",              "MARUTI",      "MARUTI.NS",      "Maruti Suzuki|Maruti"),
    ("Titan",                      "TITAN",       "TITAN.NS",       "Titan Company|Tanishq"),
    ("Sun Pharma",                 "SUNPHARMA",   "SUNPHARMA.NS",   "Sun Pharmaceutical|Sun Pharma"),
    ("Axis Bank",                  "AXISBANK",    "AXISBANK.NS",    "Axis Bank"),
    ("UltraTech Cement",           "ULTRACEMCO",  "ULTRACEMCO.NS",  "UltraTech Cement"),
    ("Nestle India",               "NESTLEIND",   "NESTLEIND.NS",   "Nestle India"),
    ("Tech Mahindra",              "TECHM",       "TECHM.NS",       "Tech Mahindra"),
    ("NTPC",                       "NTPC",        "NTPC.NS",        "NTPC"),
    ("Power Grid",                 "POWERGRID",   "POWERGRID.NS",   "Power Grid Corporation"),
    ("Tata Motors",                "TATAMOTORS",  "TATAMOTORS.NS",  "Tata Motors"),
    ("Tata Steel",                 "TATASTEEL",   "TATASTEEL.NS",   "Tata Steel"),
    ("JSW Steel",                  "JSWSTEEL",    "JSWSTEEL.NS",    "JSW Steel"),
    ("IndusInd Bank",              "INDUSINDBK",  "INDUSINDBK.NS",  "IndusInd Bank"),
    ("Dr Reddys",                  "DRREDDY",     "DRREDDY.NS",     "Dr Reddys|DRL"),
    ("Divis Labs",                 "DIVISLAB",    "DIVISLAB.NS",    "Divis Laboratories"),
    ("Cipla",                      "CIPLA",       "CIPLA.NS",       "Cipla"),
    ("Eicher Motors",              "EICHERMOT",   "EICHERMOT.NS",   "Eicher Motors|Royal Enfield"),
    ("Bajaj Auto",                 "BAJAJ-AUTO",  "BAJAJ-AUTO.NS",  "Bajaj Auto"),
    ("Hero MotoCorp",              "HEROMOTOCO",  "HEROMOTOCO.NS",  "Hero MotoCorp"),
    ("Grasim",                     "GRASIM",      "GRASIM.NS",      "Grasim Industries"),
    ("Hindalco",                   "HINDALCO",    "HINDALCO.NS",    "Hindalco Industries|Novelis"),
    ("Adani Enterprises",          "ADANIENT",    "ADANIENT.NS",    "Adani Enterprises|Adani"),
    ("Adani Ports",                "ADANIPORTS",  "ADANIPORTS.NS",  "Adani Ports|APSEZ"),
    ("Coal India",                 "COALINDIA",   "COALINDIA.NS",   "Coal India"),
    ("ONGC",                       "ONGC",        "ONGC.NS",        "ONGC"),
    ("BPCL",                       "BPCL",        "BPCL.NS",        "Bharat Petroleum|BPCL"),
    ("IOC",                        "IOC",         "IOC.NS",         "Indian Oil Corporation|IOC"),
    ("SBI Life",                   "SBILIFE",     "SBILIFE.NS",     "SBI Life Insurance"),
    ("HDFC Life",                  "HDFCLIFE",    "HDFCLIFE.NS",    "HDFC Life Insurance"),
    ("Shree Cement",               "SHREECEM",    "SHREECEM.NS",    "Shree Cement"),
    ("Apollo Hospitals",           "APOLLOHOSP",  "APOLLOHOSP.NS",  "Apollo Hospitals"),
    ("Britannia",                  "BRITANNIA",   "BRITANNIA.NS",   "Britannia Industries"),
    ("HDFC AMC",                   "HDFCAMC",     "HDFCAMC.NS",     "HDFC AMC"),
    ("Bajaj Finserv",              "BAJAJFINSV",  "BAJAJFINSV.NS",  "Bajaj Finserv"),
    ("Tata Consumer",              "TATACONSUM",  "TATACONSUM.NS",  "Tata Consumer Products"),
]

companies = pd.DataFrame(NIFTY50, columns=["company_name", "nse_symbol", "yfinance_ticker", "aliases"])
companies.to_csv(FILES["companies"], index=False)
print(f"Saved {len(companies)} companies to {FILES['companies']}")


Saved 50 companies to data\raw\nifty50_companies.csv


## 3. Collect ESG News (Fixed — Multi-Source with Fallbacks)
> **Why the original failed:** GDELT queries combining `(company_name) AND (ESG keywords)` are too 
> restrictive — most articles don't literally contain both. The paper collected from "thousands of 
> online news sources" and tagged them post-hoc.
>
> **Fix:** Three-tier approach:
> 1. **GDELT** — query company name ONLY (no ESG filter in query), then filter locally
> 2. **GNews RSS** — free fallback, no API key needed
> 3. **Synthetic scores** — generate from stock volatility + financial data if APIs fail
>
> This ensures the pipeline ALWAYS produces results for the ML/portfolio stages.

In [4]:
import requests, time, json, re, xml.etree.ElementTree as ET
from datetime import datetime, timedelta
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm

# ══════════════════════════════════════════════════════════════════════════
# METHOD 1: GDELT — Query company name ONLY, filter ESG locally
# ══════════════════════════════════════════════════════════════════════════

def test_gdelt_api():
    """Test if GDELT API is accessible."""
    try:
        resp = requests.get(
            "https://api.gdeltproject.org/api/v2/doc/doc",
            params={"query": "India economy", "mode": "artlist", 
                    "maxrecords": 3, "format": "json", "timespan": "7d"},
            timeout=15,
            headers={"User-Agent": "Mozilla/5.0 (ESG Research)"}
        )
        if resp.status_code == 200:
            data = resp.json()
            return len(data.get("articles", [])) > 0
    except:
        pass
    return False

def gdelt_collect_company(company_name, aliases, start_dt, end_dt):
    """
    FIXED: Query GDELT with just the company name (no ESG keywords).
    ESG filtering happens AFTER collection via keyword matching.
    """
    articles = []
    # Use just the primary company name — much broader results
    query_terms = [company_name.split()[0]]  # First word (e.g., "Reliance")
    if len(aliases) > 0:
        query_terms.append(aliases[0])
    
    query = " OR ".join(f'"{t}"' for t in query_terms[:2])
    
    current = start_dt
    while current < end_dt:
        chunk_end = min(current + timedelta(days=GDELT["chunk_days"]), end_dt)
        
        params = {
            "query": query,
            "mode": "artlist",
            "maxrecords": GDELT["max_records"],
            "startdatetime": current.strftime("%Y%m%d%H%M%S"),
            "enddatetime": chunk_end.strftime("%Y%m%d%H%M%S"),
            "sort": "DateDesc",
            "format": "json",
        }
        
        for attempt in range(GDELT["retries"]):
            try:
                resp = requests.get(
                    GDELT["base_url"], params=params, timeout=30,
                    headers={"User-Agent": "Mozilla/5.0 (ESG Research)"}
                )
                if resp.status_code == 200:
                    data = resp.json()
                    for a in data.get("articles", []):
                        articles.append({
                            "title": a.get("title", ""),
                            "url": a.get("url", ""),
                            "seendate": a.get("seendate", ""),
                            "domain": a.get("domain", ""),
                            "language": a.get("language", ""),
                        })
                    break
                elif resp.status_code == 403:
                    return articles  # API blocked, stop trying
            except:
                time.sleep(2 ** attempt)
        
        current = chunk_end
        time.sleep(GDELT["sleep_sec"])
    
    return articles

# ══════════════════════════════════════════════════════════════════════════
# METHOD 2: GNews RSS — Free, no API key, always works
# ══════════════════════════════════════════════════════════════════════════

def gnews_collect_company(company_name, esg_keywords):
    """Collect news via Google News RSS feed (no API key needed)."""
    articles = []
    
    # Search for company + ESG terms
    search_queries = [
        f"{company_name} ESG",
        f"{company_name} sustainability",
        f"{company_name} environment governance",
        f"{company_name} SDG climate",
    ]
    
    for query in search_queries:
        try:
            url = f"https://news.google.com/rss/search?q={requests.utils.quote(query)}&hl=en-IN&gl=IN&ceid=IN:en"
            resp = requests.get(url, timeout=15, 
                              headers={"User-Agent": "Mozilla/5.0 (ESG Research)"})
            if resp.status_code == 200:
                root = ET.fromstring(resp.content)
                for item in root.findall(".//item"):
                    title = item.findtext("title", "")
                    link = item.findtext("link", "")
                    pubdate = item.findtext("pubDate", "")
                    if title:
                        articles.append({
                            "title": title,
                            "url": link or "",
                            "seendate": pubdate,
                            "domain": "news.google.com",
                            "language": "English",
                        })
            time.sleep(1)  # Be polite
        except Exception as e:
            pass
    
    return articles

# ══════════════════════════════════════════════════════════════════════════
# METHOD 3: Synthetic ESG Scores from Financial Data
# ══════════════════════════════════════════════════════════════════════════

def generate_synthetic_news_scores(companies_df, start_date, end_date):
    """
    Generate realistic synthetic ESG sentiment scores when APIs fail.
    
    Method: Based on the paper's Table 2 statistics:
    - ESG Score: mean=0.736, SD=5.271 for SPX500-like stocks
    - SDG Score: mean=0.676, SD=5.651
    - Overall:   mean=0.928, SD=5.930
    
    We generate scores with similar distributions, adding company-specific
    bias based on sector (tech/green companies get higher ESG scores).
    """
    np.random.seed(42)
    
    dates = pd.date_range(start_date, end_date, freq='B')  # business days
    
    # Company ESG bias (higher = greener sector)
    green_keywords = ["power", "energy", "solar", "ntpc", "grid", "coal"]
    tech_keywords = ["tcs", "infy", "wipro", "hcl", "tech"]
    
    records = []
    for _, company in companies_df.iterrows():
        symbol = company["nse_symbol"].lower()
        
        # Sector-based ESG bias
        if any(k in symbol for k in green_keywords):
            esg_bias = 2.0
        elif any(k in symbol for k in tech_keywords):
            esg_bias = 1.5
        else:
            esg_bias = 0.5
        
        for date in dates:
            # Generate scores matching paper's distribution (Table 2)
            sentiment = np.clip(np.random.normal(0.93 + esg_bias * 0.1, 5.93), -100, 100)
            esg_score = np.clip(np.random.normal(0.74 + esg_bias, 5.27), -100, 100)
            sdg_score = np.clip(np.random.normal(0.68 + esg_bias * 0.8, 5.65), -100, 100)
            polarity = abs(np.random.normal(20, 15))
            
            # Some days have no news (random ~30% missing)
            if np.random.random() < 0.3:
                continue
            
            records.append({
                "nse_symbol": company["nse_symbol"],
                "yfinance_ticker": company["yfinance_ticker"],
                "company_name": company["company_name"],
                "title": f"Synthetic ESG article for {company['company_name']}",
                "url": f"synthetic://{company['nse_symbol']}/{date.strftime('%Y%m%d')}",
                "seendate": date.strftime("%Y%m%dT%H%M%SZ"),
                "domain": "synthetic",
                "language": "English",
                "article_date": date,
                "sentiment_score": round(sentiment, 2),
                "positive_tone": round(max(0, sentiment) * 0.6, 2),
                "negative_tone": round(abs(min(0, sentiment)) * 0.6, 2),
                "polarity": round(polarity, 2),
                "esg_score_raw": round(max(0, esg_score / 10), 2),
                "sdg_score_raw": round(max(0, sdg_score / 10), 2),
                "is_synthetic": True,
            })
    
    return pd.DataFrame(records)

# ══════════════════════════════════════════════════════════════════════════
# MAIN COLLECTION LOGIC
# ══════════════════════════════════════════════════════════════════════════

out_path = FILES["raw_news"]
companies = pd.read_csv(FILES["companies"])

if out_path.exists() and pd.read_csv(out_path).shape[0] > 100:
    print(f"Raw news already exists with {pd.read_csv(out_path).shape[0]} rows. Loading...")
    raw_df = pd.read_csv(out_path)
else:
    print("=" * 70)
    print("STEP 1: Testing GDELT API connectivity...")
    print("=" * 70)
    gdelt_works = test_gdelt_api()
    
    if gdelt_works:
        print("✓ GDELT API is accessible. Using GDELT (company-only query).")
        method_used = "GDELT"
    else:
        print("✗ GDELT API is not accessible (403/timeout).")
        print("  Trying Google News RSS fallback...")
        method_used = "GNews"
    
    all_articles = []
    
    if method_used == "GDELT":
        # ── GDELT Collection (fixed: company name only, no ESG in query) ──
        print(f"\nCollecting from GDELT for {len(companies)} companies...")
        print("NOTE: Querying company name only → filtering ESG locally\n")
        
        for _, row in tqdm(companies.iterrows(), total=len(companies), desc="GDELT collection"):
            aliases = [a.strip() for a in str(row["aliases"]).split("|")]
            arts = gdelt_collect_company(
                row["company_name"], aliases, START_DATE, END_DATE
            )
            for a in arts:
                a["nse_symbol"] = row["nse_symbol"]
                a["yfinance_ticker"] = row["yfinance_ticker"]
                a["company_name"] = row["company_name"]
            all_articles.extend(arts)
            
            if len(arts) == 0 and method_used == "GDELT":
                # GDELT might be blocking — switch to GNews
                print(f"  {row['nse_symbol']}: 0 articles from GDELT, trying GNews...")
                gnews_arts = gnews_collect_company(row["company_name"], ESG_KEYWORDS[:3])
                for a in gnews_arts:
                    a["nse_symbol"] = row["nse_symbol"]
                    a["yfinance_ticker"] = row["yfinance_ticker"]
                    a["company_name"] = row["company_name"]
                all_articles.extend(gnews_arts)
                if gnews_arts:
                    print(f"    → Got {len(gnews_arts)} from GNews RSS")
                    method_used = "GNews+GDELT"
            else:
                print(f"  {row['nse_symbol']}: {len(arts)} articles")
    
    if method_used == "GNews" or len(all_articles) < 50:
        # ── GNews RSS Fallback ──
        print(f"\nCollecting from Google News RSS for {len(companies)} companies...")
        for _, row in tqdm(companies.iterrows(), total=len(companies), desc="GNews RSS"):
            arts = gnews_collect_company(row["company_name"], ESG_KEYWORDS[:3])
            for a in arts:
                a["nse_symbol"] = row["nse_symbol"]
                a["yfinance_ticker"] = row["yfinance_ticker"]
                a["company_name"] = row["company_name"]
            all_articles.extend(arts)
            if len(arts) > 0:
                print(f"  {row['nse_symbol']}: {len(arts)} articles")
    
    if len(all_articles) < 50:
        # ── Synthetic Fallback ──
        print("\n" + "=" * 70)
        print("⚠ APIs returned insufficient data. Generating synthetic ESG scores.")
        print("  (Based on paper's Table 2 distribution statistics)")
        print("  This allows the ML pipeline to run and demonstrate the methodology.")
        print("=" * 70)
        
        synth_df = generate_synthetic_news_scores(companies, START_DATE, END_DATE)
        synth_df.to_csv(out_path, index=False)
        
        # Also save directly as scored news (skip NLP scoring cell)
        synth_df.to_csv(FILES["scored_news"], index=False)
        
        raw_df = synth_df
        print(f"\nGenerated {len(raw_df):,} synthetic daily score records")
        print(f"Companies: {raw_df['nse_symbol'].nunique()}")
        print(f"Date range: {raw_df['article_date'].min()} to {raw_df['article_date'].max()}")
        
        # Flag to skip the NLP scoring cell
        USING_SYNTHETIC = True
    else:
        raw_df = pd.DataFrame(all_articles)
        raw_df = raw_df.drop_duplicates(subset=["nse_symbol", "url"])
        raw_df.to_csv(out_path, index=False)
        USING_SYNTHETIC = False
        print(f"\nTotal articles collected: {len(raw_df):,}")
        print(f"Method: {method_used}")

    print(f"Companies covered: {raw_df['nse_symbol'].nunique() if 'nse_symbol' in raw_df.columns else 'N/A'}")

try:
    _ = USING_SYNTHETIC
except NameError:
    USING_SYNTHETIC = False

print(f"\nUsing synthetic data: {USING_SYNTHETIC}")
print(f"Raw data shape: {raw_df.shape}")


Raw news already exists with 8755 rows. Loading...

Using synthetic data: False
Raw data shape: (8755, 8)


## 4. Clean News + NLP Scoring (Vectorized)
> Combines cleaning and scoring in one pass. Uses vectorized string operations instead of iterrows.
> Follows the paper's lexicon-based approach with sentiment scores from -100 to +100.

In [5]:
# ============================================================
# 4. CLEAN NEWS + NLP SCORING — ROBUST CELL
# ============================================================

import re
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from IPython.display import display
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

# Important: after kernel restart this variable may not exist.
# So we safely define it here.
USING_SYNTHETIC = globals().get("USING_SYNTHETIC", False)

print(f"USING_SYNTHETIC = {USING_SYNTHETIC}")

if USING_SYNTHETIC:
    print("Using synthetic scores — skipping NLP scoring step.")
    scored_df = pd.read_csv(FILES["scored_news"])

    if "article_date" not in scored_df.columns:
        date_source = scored_df["seendate"] if "seendate" in scored_df.columns else pd.NaT
        scored_df["article_date"] = pd.to_datetime(date_source, errors="coerce")
    else:
        scored_df["article_date"] = pd.to_datetime(scored_df["article_date"], errors="coerce")

    scored_df.to_csv(FILES["scored_news"], index=False)
    print(f"Synthetic scored articles: {len(scored_df):,}")

else:
    if not FILES["raw_news"].exists():
        raise FileNotFoundError(f"Raw news file not found: {FILES['raw_news']} — run news collection cell first.")

    vader = SentimentIntensityAnalyzer()
    raw_df = pd.read_csv(FILES["raw_news"])

    required = {"title", "url", "nse_symbol", "yfinance_ticker", "company_name"}
    missing = required - set(raw_df.columns)
    if missing:
        raise ValueError(f"Raw news file is missing columns: {missing}")

    df = raw_df.dropna(subset=["title", "url"]).copy()
    df["title"] = df["title"].astype(str).str.strip()
    df = df[df["title"].str.len() > 5]

    def parse_any_date(x):
        if pd.isna(x):
            return pd.NaT
        # Handles GDELT format, Google RSS pubDate, and normal datetime strings
        dt = pd.to_datetime(x, errors="coerce", utc=True)
        if pd.notna(dt):
            return dt.tz_convert(None) if hasattr(dt, "tz_convert") else dt
        s = str(x).replace("T", "").replace("Z", "").replace("-", "").replace(":", "")[:14]
        return pd.to_datetime(s, format="%Y%m%d%H%M%S", errors="coerce")

    df["article_date"] = df["seendate"].apply(parse_any_date) if "seendate" in df.columns else pd.NaT
    df = df.dropna(subset=["article_date"])
    df = df.drop_duplicates(subset=["nse_symbol", "url"])
    df = df.sort_values(["nse_symbol", "article_date"]).reset_index(drop=True)

    print(f"Cleaned articles: {len(df):,}")
    FILES["cleaned_news"].parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(FILES["cleaned_news"], index=False)

    def make_pattern(keywords):
        terms = [re.escape(str(k).lower().strip()) for k in keywords if str(k).strip()]
        return re.compile("|".join(terms)) if terms else re.compile(r"$^")

    pat_env  = make_pattern(KW_ENVIRONMENTAL)
    pat_soc  = make_pattern(KW_SOCIAL)
    pat_gov  = make_pattern(KW_GOVERNANCE)
    pat_sdg  = make_pattern(KW_SDG)
    pat_cont = make_pattern(KW_CONTROVERSY)

    text_lower = df["title"].fillna("").astype(str).str.lower()

    print("Running VADER sentiment scoring...")
    vader_scores = [vader.polarity_scores(t) for t in tqdm(df["title"].fillna("").astype(str), desc="VADER")]
    vader_df = pd.DataFrame(vader_scores)

    df["sentiment_score"] = (vader_df["compound"] * 100).round(2)
    df["positive_tone"]   = (vader_df["pos"] * 100).round(2)
    df["negative_tone"]   = (vader_df["neg"] * 100).round(2)
    df["neutral_tone"]    = (vader_df["neu"] * 100).round(2)
    df["polarity"]        = (df["positive_tone"].abs() + df["negative_tone"].abs()).round(2)

    print("Scoring ESG/SDG keywords...")
    df["env_score"]    = text_lower.str.count(pat_env)
    df["social_score"] = text_lower.str.count(pat_soc)
    df["gov_score"]    = text_lower.str.count(pat_gov)
    df["sdg_score"]    = text_lower.str.count(pat_sdg)
    df["esg_score"]    = df["env_score"] + df["social_score"] + df["gov_score"]
    df["controversy"]  = (text_lower.str.count(pat_cont) > 0).astype(int)

    FILES["scored_news"].parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(FILES["scored_news"], index=False)
    scored_df = df.copy()

    print(f"Scored articles saved: {len(scored_df):,}")
    display(scored_df[["sentiment_score", "positive_tone", "negative_tone", "esg_score", "sdg_score", "controversy"]].describe().round(2))


USING_SYNTHETIC = False
Cleaned articles: 8,755
Running VADER sentiment scoring...


VADER: 100%|██████████| 8755/8755 [00:00<00:00, 11563.29it/s]


Scoring ESG/SDG keywords...
Scored articles saved: 8,755


,sentiment_score,positive_tone,negative_tone,esg_score,sdg_score,controversy
count,8755.00,8755.00,8755.00,8755.00,8755.00,8755.00
mean,18.43,10.89,2.22,0.60,0.02,0.01
std,31.66,12.98,6.35,0.82,0.17,0.12
min,-87.20,0.00,0.00,0.00,0.00,0.00
25%,0.00,0.00,0.00,0.00,0.00,0.00
50%,0.00,6.90,0.00,0.00,0.00,0.00
75%,42.15,18.90,0.00,1.00,0.00,0.00
max,96.48,67.70,65.80,6.00,3.00,1.00


## 5. Aggregate to Daily Company Scores
> Creates three sentiment indicators per the paper: ESG score, SDG score, Overall sentiment score.  
> These become the stock selection criteria.

In [6]:
# ============================================================
# 5. AGGREGATE TO DAILY COMPANY SCORES — ROBUST CELL
# ============================================================

import numpy as np
import pandas as pd
from IPython.display import display

USING_SYNTHETIC = globals().get("USING_SYNTHETIC", False)

if not FILES["scored_news"].exists():
    raise FileNotFoundError(f"Scored news file not found: {FILES['scored_news']} — run NLP scoring cell first.")

scored = pd.read_csv(FILES["scored_news"])

# Robust date creation
if "article_date" in scored.columns:
    scored["date"] = pd.to_datetime(scored["article_date"], errors="coerce").dt.normalize()
elif "seendate" in scored.columns:
    scored["date"] = pd.to_datetime(scored["seendate"], errors="coerce", utc=True).dt.tz_convert(None).dt.normalize()
else:
    raise ValueError("Neither article_date nor seendate exists in scored_news.")

scored = scored.dropna(subset=["date"])

# Standardize synthetic/raw score column names
rename_map = {}
if "esg_score_raw" in scored.columns and "esg_score" not in scored.columns:
    rename_map["esg_score_raw"] = "esg_score"
if "sdg_score_raw" in scored.columns and "sdg_score" not in scored.columns:
    rename_map["sdg_score_raw"] = "sdg_score"
scored = scored.rename(columns=rename_map)

# Ensure required score columns exist
for col in ["sentiment_score", "positive_tone", "negative_tone", "polarity", "esg_score", "sdg_score", "env_score", "social_score", "gov_score", "controversy"]:
    if col not in scored.columns:
        scored[col] = 0

# Ensure ID columns exist
for col in ["nse_symbol", "yfinance_ticker", "company_name"]:
    if col not in scored.columns:
        raise ValueError(f"Missing required column in scored_news: {col}")

# Add article counter before grouping
scored["article_id_count"] = 1

agg_funcs = {
    "sentiment_score": "mean",
    "positive_tone": "mean",
    "negative_tone": "mean",
    "polarity": "mean",
    "esg_score": "sum",
    "sdg_score": "sum",
    "env_score": "sum",
    "social_score": "sum",
    "gov_score": "sum",
    "controversy": "sum",
    "article_id_count": "sum",
}

daily = (
    scored.groupby(["nse_symbol", "yfinance_ticker", "company_name", "date"], as_index=False)
          .agg(agg_funcs)
          .rename(columns={"article_id_count": "article_count", "controversy": "controversy_count"})
)

# Paper-style three indicators
daily["overall_score"]  = daily["sentiment_score"]
daily["esg_sent_score"] = daily["sentiment_score"] * (1 + daily["esg_score"].clip(lower=0) / 10)
daily["sdg_sent_score"] = daily["sentiment_score"] * (1 + daily["sdg_score"].clip(lower=0) / 10)
daily["controversy_ratio"] = (daily["controversy_count"] / daily["article_count"].replace(0, np.nan)).fillna(0)

daily.to_csv(FILES["daily_scores"], index=False)

print(f"Daily scores shape: {daily.shape}")
print(f"Date range: {daily['date'].min()} to {daily['date'].max()}")
print(f"Companies covered: {daily['nse_symbol'].nunique()}")
display(daily.head())


Daily scores shape: (7150, 19)
Date range: 1999-07-31 00:00:00 to 2026-05-06 00:00:00
Companies covered: 50


,nse_symbol,yfinance_ticker,company_name,date,sentiment_score,positive_tone,negative_tone,polarity,esg_score,sdg_score,env_score,social_score,gov_score,controversy_count,article_count,overall_score,esg_sent_score,sdg_sent_score,controversy_ratio
0,ADANIENT,ADANIENT.NS,Adani Enterprises,2013-04-18,-52.67,0.0,19.5,19.5,1,0,1,0,0,0,1,-52.67,-57.937,-52.67,0.0
1,ADANIENT,ADANIENT.NS,Adani Enterprises,2013-09-05,0.00,0.0,0.0,0.0,0,0,0,0,0,0,1,0.00,0.000,0.00,0.0
2,ADANIENT,ADANIENT.NS,Adani Enterprises,2017-08-15,2.58,7.3,0.0,7.3,0,0,0,0,0,0,1,2.58,2.580,2.58,0.0
3,ADANIENT,ADANIENT.NS,Adani Enterprises,2018-02-28,-5.16,16.0,13.1,29.1,0,0,0,0,0,0,1,-5.16,-5.160,-5.16,0.0
4,ADANIENT,ADANIENT.NS,Adani Enterprises,2019-01-30,0.00,0.0,0.0,0.0,1,0,1,0,0,0,1,0.00,0.000,0.00,0.0


## 6. Download Stock Data (yfinance)

In [7]:
# ============================================================
# 6. DOWNLOAD STOCK DATA — ROBUST YFINANCE CELL
# ============================================================

import numpy as np
import pandas as pd
import yfinance as yf
from IPython.display import display

companies = pd.read_csv(FILES["companies"])
tickers = companies["yfinance_ticker"].dropna().unique().tolist()
benchmark = "^NSEI"
out_path = FILES["stock_returns"]

if out_path.exists():
    print("Stock returns already exist. Loading...")
    stock_returns = pd.read_csv(out_path, parse_dates=["date"])
else:
    print(f"Downloading {len(tickers)} tickers + benchmark from yfinance...")
    all_tickers = tickers + [benchmark]
    data = yf.download(
        all_tickers,
        start=YFINANCE_START,
        end=YFINANCE_END,
        auto_adjust=True,
        progress=True,
        threads=True,
        group_by="column"
    )

    if data.empty:
        raise RuntimeError("yfinance returned empty data. Check internet connection/date range/tickers.")

    if isinstance(data.columns, pd.MultiIndex):
        # With auto_adjust=True, Close is adjusted close
        close = data["Close"].copy()
    else:
        close = data[["Close"]].copy()
        close.columns = [tickers[0]]

    close = close.dropna(how="all")
    returns = close.pct_change()

    if benchmark in returns.columns:
        benchmark_ret = returns[benchmark].fillna(0)
    else:
        benchmark_ret = pd.Series(0, index=returns.index)

    frames = []
    for ticker in tickers:
        if ticker not in returns.columns:
            print(f"Warning: no yfinance data for {ticker}")
            continue
        temp = pd.DataFrame({
            "date": returns.index,
            "yfinance_ticker": ticker,
            "daily_return": returns[ticker].values,
            "adj_close": close[ticker].values if ticker in close.columns else np.nan,
            "benchmark_return": benchmark_ret.values,
        })
        temp["return_lag1"] = temp["daily_return"].shift(1)
        temp["return_lag2"] = temp["daily_return"].shift(2)
        temp["return_lag5"] = temp["daily_return"].shift(5)
        temp["rolling_vol_20"] = temp["daily_return"].rolling(20, min_periods=10).std() * np.sqrt(252)
        temp["excess_return"] = temp["daily_return"] - temp["benchmark_return"]
        frames.append(temp)

    if not frames:
        raise RuntimeError("No ticker returned usable yfinance data.")

    stock_returns = pd.concat(frames, ignore_index=True).dropna(subset=["daily_return"])

    ticker_info = companies.set_index("yfinance_ticker")[["company_name", "nse_symbol"]]
    stock_returns = stock_returns.merge(ticker_info, left_on="yfinance_ticker", right_index=True, how="left")

    out_path.parent.mkdir(parents=True, exist_ok=True)
    stock_returns.to_csv(out_path, index=False)

print(f"Stock returns shape: {stock_returns.shape}")
print(f"Tickers with data: {stock_returns['yfinance_ticker'].nunique()}")
display(stock_returns.head())


Stock returns already exist. Loading...
Stock returns shape: (12103, 12)
Tickers with data: 49


,date,yfinance_ticker,daily_return,adj_close,benchmark_return,return_lag1,return_lag2,return_lag5,rolling_vol_20,excess_return,company_name,nse_symbol
0,2025-05-08,RELIANCE.NS,0.000711,1401.402954,-0.005759,NaN,NaN,NaN,NaN,0.006470,Reliance Industries,RELIANCE
1,2025-05-09,RELIANCE.NS,-0.021180,1371.721436,-0.010950,0.000711,NaN,NaN,NaN,-0.010230,Reliance Industries,RELIANCE
2,2025-05-12,RELIANCE.NS,0.043058,1430.785522,0.038183,-0.021180,0.000711,NaN,NaN,0.004875,Reliance Industries,RELIANCE
3,2025-05-13,RELIANCE.NS,-0.014480,1410.068237,-0.013896,0.043058,-0.021180,NaN,NaN,-0.000584,Reliance Industries,RELIANCE
4,2025-05-14,RELIANCE.NS,0.006145,1418.733765,0.003603,-0.014480,0.043058,NaN,NaN,0.002543,Reliance Industries,RELIANCE


## 7. Merge News Scores + Stock Returns → Model Dataset

In [8]:
# ============================================================
# 7. MERGE NEWS SCORES + STOCK RETURNS — ROBUST CELL
# ============================================================

import pandas as pd
from IPython.display import display

if not FILES["daily_scores"].exists():
    raise FileNotFoundError(f"Daily scores not found: {FILES['daily_scores']}")
if not FILES["stock_returns"].exists():
    raise FileNotFoundError(f"Stock returns not found: {FILES['stock_returns']}")

daily_scores = pd.read_csv(FILES["daily_scores"], parse_dates=["date"])
stock_returns = pd.read_csv(FILES["stock_returns"], parse_dates=["date"])

# Normalize dates on both sides to avoid time-component mismatch
daily_scores["date"] = pd.to_datetime(daily_scores["date"], errors="coerce").dt.normalize()
stock_returns["date"] = pd.to_datetime(stock_returns["date"], errors="coerce").dt.normalize()

model_df = stock_returns.merge(
    daily_scores,
    on=["yfinance_ticker", "date"],
    how="left",
    suffixes=("", "_news")
)

news_cols = [
    "sentiment_score", "positive_tone", "negative_tone", "polarity",
    "esg_score", "sdg_score", "env_score", "social_score", "gov_score",
    "article_count", "controversy_count", "controversy_ratio",
    "overall_score", "esg_sent_score", "sdg_sent_score"
]

for col in news_cols:
    if col not in model_df.columns:
        model_df[col] = 0
    model_df[col] = model_df[col].fillna(0)

# Resolve duplicate identity columns
for col in ["company_name", "nse_symbol"]:
    news_col = f"{col}_news"
    if news_col in model_df.columns:
        model_df[col] = model_df[col].combine_first(model_df[news_col])
        model_df = model_df.drop(columns=[news_col])

model_df = model_df.sort_values(["yfinance_ticker", "date"]).reset_index(drop=True)
model_df["next_day_return"] = model_df.groupby("yfinance_ticker")["daily_return"].shift(-1)

FILES["model_dataset"].parent.mkdir(parents=True, exist_ok=True)
model_df.to_csv(FILES["model_dataset"], index=False)

print(f"Model dataset shape: {model_df.shape}")
print(f"Date range: {model_df['date'].min()} to {model_df['date'].max()}")
print(f"Tickers: {model_df['yfinance_ticker'].nunique()}")
print(f"Rows with news: {(model_df['article_count'] > 0).sum():,} / {len(model_df):,}")
display(model_df.head())


Model dataset shape: (12103, 28)
Date range: 2025-05-08 00:00:00 to 2026-05-05 00:00:00
Tickers: 49
Rows with news: 2,427 / 12,103


,date,yfinance_ticker,daily_return,adj_close,benchmark_return,return_lag1,return_lag2,return_lag5,rolling_vol_20,excess_return,...,env_score,social_score,gov_score,controversy_count,article_count,overall_score,esg_sent_score,sdg_sent_score,controversy_ratio,next_day_return
0,2025-05-08,ADANIENT.NS,-0.028813,2284.167725,-0.005759,NaN,NaN,NaN,NaN,-0.023054,...,0.0,0.0,0.0,0.0,0.0,0.000,0.000,0.000,0.0,-0.015009
1,2025-05-09,ADANIENT.NS,-0.015009,2249.884766,-0.010950,-0.028813,NaN,NaN,NaN,-0.004059,...,0.0,0.0,0.0,0.0,0.0,0.000,0.000,0.000,0.0,0.077432
2,2025-05-12,ADANIENT.NS,0.077432,2424.098389,0.038183,-0.015009,-0.028813,NaN,NaN,0.039249,...,0.0,0.0,0.0,0.0,2.0,8.895,8.895,8.895,0.0,0.006556
3,2025-05-13,ADANIENT.NS,0.006556,2439.990234,-0.013896,0.077432,-0.015009,NaN,NaN,0.020452,...,0.0,0.0,0.0,0.0,0.0,0.000,0.000,0.000,0.0,0.008479
4,2025-05-14,ADANIENT.NS,0.008479,2460.679932,0.003603,0.006556,0.077432,NaN,NaN,0.004877,...,0.0,0.0,0.0,0.0,0.0,0.000,0.000,0.000,0.0,0.023600


## 8. ML Models + Portfolio Backtest
> 1. Chronological 70/30 train/test split (no shuffling, so there is no look-ahead from random splits)
> 2. Features: daily sentiment, ESG/SDG/controversy scores, article counts, lagged returns and 20-day rolling volatility
> 3. Target: next-day stock return
> 4. Each month in the test period, pick the top 10 stocks by mean predicted return and hold them equally weighted
> 5. Benchmark: equal-weighted portfolio of all stocks over the same months

**Models:** Linear Regression, LASSO, Random Forest, Gradient Boosting


In [9]:
# ============================================================
# 8. SIMPLE ROBUST ML + PORTFOLIO BACKTEST
# This replaces the very heavy/error-prone paper-style cell.
# It works even with small 5-company samples.
# ============================================================

import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from IPython.display import display

model_df = pd.read_csv(FILES["model_dataset"], parse_dates=["date"])
model_df = model_df.sort_values(["date", "yfinance_ticker"]).reset_index(drop=True)

feature_cols = [
    "sentiment_score", "positive_tone", "negative_tone", "polarity",
    "esg_score", "sdg_score", "env_score", "social_score", "gov_score",
    "article_count", "controversy_count", "controversy_ratio",
    "overall_score", "esg_sent_score", "sdg_sent_score",
    "return_lag1", "return_lag2", "return_lag5", "rolling_vol_20"
]
feature_cols = [c for c in feature_cols if c in model_df.columns]

ml_df = model_df.dropna(subset=["next_day_return"]).copy()
for c in feature_cols:
    ml_df[c] = pd.to_numeric(ml_df[c], errors="coerce").fillna(0)

if len(ml_df) < 50:
    raise RuntimeError(f"Too few rows for ML/backtest: {len(ml_df)}. Check previous outputs.")

# Time split, not random split
unique_dates = sorted(ml_df["date"].unique())
split_date = unique_dates[int(len(unique_dates) * 0.70)]
train = ml_df[ml_df["date"] <= split_date].copy()
test  = ml_df[ml_df["date"] > split_date].copy()

X_train, y_train = train[feature_cols], train["next_day_return"]
X_test,  y_test  = test[feature_cols],  test["next_day_return"]

models = {
    "Linear": LinearRegression(),
    "LASSO": Lasso(alpha=0.001, max_iter=10000),
    "RF": RandomForestRegressor(n_estimators=100, max_depth=6, random_state=42, n_jobs=-1),
    "GB": GradientBoostingRegressor(random_state=42)
}

pred_frames = []
perf_rows = []

for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)

    temp = test[["date", "yfinance_ticker", "nse_symbol", "company_name", "daily_return", "next_day_return"]].copy()
    temp["model"] = name
    temp["predicted_return"] = pred
    pred_frames.append(temp)

    perf_rows.append({
        "Model": name,
        "RMSE": np.sqrt(mean_squared_error(y_test, pred)),
        "MAE": mean_absolute_error(y_test, pred),
        "R2": r2_score(y_test, pred),
        "Train_Rows": len(train),
        "Test_Rows": len(test)
    })

predictions = pd.concat(pred_frames, ignore_index=True)
performance = pd.DataFrame(perf_rows)

FILES["model_performance"].parent.mkdir(parents=True, exist_ok=True)
performance.to_csv(FILES["model_performance"], index=False)
predictions.to_csv(PATHS["outputs_models"] / "model_predictions.csv", index=False)

print("Model performance:")
display(performance.round(6))

# ---------------- Portfolio Backtest ----------------
# Monthly rebalance: each month choose top stocks by predicted return.
# For small 5-company sample, use top 3. For larger sample, top 10.

top_n = 3 if predictions["yfinance_ticker"].nunique() <= 10 else 10
port_rows = []

for model_name, grp_model in predictions.groupby("model"):
    grp_model = grp_model.copy()
    grp_model["month"] = grp_model["date"].dt.to_period("M")

    for month, grp in grp_model.groupby("month"):
        # Use mean predicted return within the month to select stocks
        ranking = grp.groupby("yfinance_ticker")["predicted_return"].mean().sort_values(ascending=False)
        selected = ranking.head(min(top_n, len(ranking))).index.tolist()
        if not selected:
            continue

        month_data = grp[grp["yfinance_ticker"].isin(selected)]
        stock_month_returns = month_data.groupby("yfinance_ticker")["daily_return"].apply(lambda x: (1 + x).prod() - 1)
        monthly_return = stock_month_returns.mean()

        port_rows.append({
            "window": "70_30_split",
            "top_m": len(selected),
            "score_type": "ml_predicted_return",
            "model": model_name,
            "method": "TopN_EqualWeight",
            "rebalance_date": str(month),
            "monthly_return": monthly_return
        })

# Equal-weight benchmark on the same test period
test_eq = test.copy()
test_eq["month"] = test_eq["date"].dt.to_period("M")
for month, grp in test_eq.groupby("month"):
    stock_month_returns = grp.groupby("yfinance_ticker")["daily_return"].apply(lambda x: (1 + x).prod() - 1)
    port_rows.append({
        "window": "70_30_split",
        "top_m": stock_month_returns.shape[0],
        "score_type": "equal_weight",
        "model": "Equal",
        "method": "EqualWeight",
        "rebalance_date": str(month),
        "monthly_return": stock_month_returns.mean()
    })

port_ret_df = pd.DataFrame(port_rows)
FILES["portfolio_returns"].parent.mkdir(parents=True, exist_ok=True)
port_ret_df.to_csv(FILES["portfolio_returns"], index=False)

print(f"Portfolio return observations: {len(port_ret_df):,}")
display(port_ret_df.head())


Model performance:


,Model,RMSE,MAE,R2,Train_Rows,Test_Rows
0,Linear,0.018985,0.013819,-0.013509,8477,3577
1,LASSO,0.018877,0.013726,-0.002017,8477,3577
2,RF,0.018866,0.013735,-0.000898,8477,3577
3,GB,0.019056,0.013885,-0.021114,8477,3577


Portfolio return observations: 25


,window,top_m,score_type,model,method,rebalance_date,monthly_return
0,70_30_split,10,ml_predicted_return,GB,TopN_EqualWeight,2026-01,0.002856
1,70_30_split,10,ml_predicted_return,GB,TopN_EqualWeight,2026-02,0.048423
2,70_30_split,10,ml_predicted_return,GB,TopN_EqualWeight,2026-03,-0.132954
3,70_30_split,10,ml_predicted_return,GB,TopN_EqualWeight,2026-04,0.132480
4,70_30_split,10,ml_predicted_return,GB,TopN_EqualWeight,2026-05,0.000473


## 9. Performance Summary — Annualized Sharpe Ratios
> Annualized return, volatility and Sharpe ratio (6.5% risk-free rate) for each model's portfolio and the benchmark.


In [10]:
# ============================================================
# 9. PERFORMANCE SUMMARY — ROBUST CELL
# ============================================================

import numpy as np
import pandas as pd
from IPython.display import display

if not FILES["portfolio_returns"].exists():
    raise FileNotFoundError(f"Portfolio returns not found: {FILES['portfolio_returns']} — run ML/backtest cell first.")

port_ret_df = pd.read_csv(FILES["portfolio_returns"])

required_cols = {"model", "method", "monthly_return"}
if port_ret_df.empty or not required_cols.issubset(port_ret_df.columns):
    print("No valid portfolio results available.")
    perf_df = pd.DataFrame(columns=["Model", "Method", "Ann_Return", "Ann_Vol", "Sharpe", "N_Months"])
else:
    td = PORTFOLIO["trading_days"]
    rfr = PORTFOLIO["risk_free_rate"]
    results = []

    group_cols = [c for c in ["window", "top_m", "score_type", "model", "method"] if c in port_ret_df.columns]
    for keys, grp in port_ret_df.groupby(group_cols):
        rets = pd.to_numeric(grp["monthly_return"], errors="coerce").dropna()
        if len(rets) < 2:
            continue
        ann_ret = (1 + rets.mean()) ** 12 - 1
        ann_vol = rets.std() * np.sqrt(12)
        sharpe = (ann_ret - rfr) / ann_vol if ann_vol > 0 else np.nan

        row = dict(zip(group_cols, keys if isinstance(keys, tuple) else (keys,)))
        row.update({
            "Ann_Return": round(ann_ret, 4),
            "Ann_Vol": round(ann_vol, 4),
            "Sharpe": round(sharpe, 4) if pd.notna(sharpe) else np.nan,
            "N_Months": len(rets)
        })
        results.append(row)

    perf_df = pd.DataFrame(results)

FILES["portfolio_perf"].parent.mkdir(parents=True, exist_ok=True)
perf_df.to_csv(FILES["portfolio_perf"], index=False)

print("Portfolio performance summary:")
display(perf_df.sort_values("Sharpe", ascending=False).round(4) if len(perf_df) else perf_df)


Portfolio performance summary:


,window,top_m,score_type,model,method,Ann_Return,Ann_Vol,Sharpe,N_Months
3,70_30_split,10,ml_predicted_return,RF,TopN_EqualWeight,0.2650,0.3214,0.6221,5
0,70_30_split,10,ml_predicted_return,GB,TopN_EqualWeight,0.1303,0.3334,0.1957,5
1,70_30_split,10,ml_predicted_return,LASSO,TopN_EqualWeight,0.0622,0.3570,-0.0079,5
4,70_30_split,49,equal_weight,Equal,EqualWeight,-0.0101,0.2474,-0.3035,5
2,70_30_split,10,ml_predicted_return,Linear,TopN_EqualWeight,-0.1799,0.1654,-1.4810,5


## 10. Visualizations

In [11]:
# ============================================================
# 10. VISUALIZATIONS — ROBUST CELL
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

FIG_DIR = PATHS["outputs_figures"]
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Plot 1: Cumulative portfolio returns
if FILES["portfolio_returns"].exists():
    port_ret_df = pd.read_csv(FILES["portfolio_returns"])
    if not port_ret_df.empty and {"rebalance_date", "model", "monthly_return"}.issubset(port_ret_df.columns):
        plt.figure(figsize=(10, 5))
        for model, grp in port_ret_df.groupby("model"):
            g = grp.sort_values("rebalance_date")
            cum = (1 + g.groupby("rebalance_date")["monthly_return"].mean()).cumprod()
            plt.plot(range(len(cum)), cum.values, label=model)
        plt.title("Cumulative Portfolio Returns")
        plt.xlabel("Month")
        plt.ylabel("Cumulative return")
        plt.legend()
        plt.tight_layout()
        plt.savefig(FIG_DIR / "cumulative_returns.png", dpi=150)
        plt.close()
        print("Saved cumulative_returns.png")

# Plot 2: Article count by company
if FILES["daily_scores"].exists():
    daily_scores = pd.read_csv(FILES["daily_scores"])
    if "company_name" in daily_scores.columns and "article_count" in daily_scores.columns:
        article_counts = daily_scores.groupby("company_name")["article_count"].sum().sort_values(ascending=True)
        plt.figure(figsize=(8, max(4, len(article_counts) * 0.35)))
        article_counts.plot(kind="barh")
        plt.title("Article Count by Company")
        plt.xlabel("Articles")
        plt.tight_layout()
        plt.savefig(FIG_DIR / "article_count_by_company.png", dpi=150)
        plt.close()
        print("Saved article_count_by_company.png")

# Plot 3: Model performance
if FILES["portfolio_perf"].exists():
    perf_df = pd.read_csv(FILES["portfolio_perf"])
    if not perf_df.empty and {"model", "Sharpe"}.issubset(perf_df.columns):
        avg_sharpe = perf_df.groupby("model")["Sharpe"].mean().sort_values()
        plt.figure(figsize=(8, 5))
        avg_sharpe.plot(kind="barh")
        plt.title("Average Sharpe by Model")
        plt.xlabel("Sharpe ratio")
        plt.tight_layout()
        plt.savefig(FIG_DIR / "model_ranking.png", dpi=150)
        plt.close()
        print("Saved model_ranking.png")

print(f"Figures saved to: {FIG_DIR}")


Saved cumulative_returns.png
Saved article_count_by_company.png
Saved model_ranking.png
Figures saved to: outputs\figures


## 11. Results Summary

In [12]:
# ============================================================
# 11. FINAL RESULTS SUMMARY — ROBUST CELL
# ============================================================

import pandas as pd
from IPython.display import display

print("=" * 80)
print("FINAL RESULTS SUMMARY")
print("=" * 80)

if FILES["portfolio_perf"].exists():
    perf = pd.read_csv(FILES["portfolio_perf"])
    print(f"\nTotal performance rows: {len(perf)}")
    if not perf.empty and "Sharpe" in perf.columns:
        print("\n--- Top Configurations by Sharpe Ratio ---")
        display(perf.sort_values("Sharpe", ascending=False).head(10))
        if "model" in perf.columns:
            print("\n--- Average Sharpe by Model ---")
            display(perf.groupby("model")["Sharpe"].mean().sort_values(ascending=False).round(4))
    else:
        print("Performance file exists, but it has no valid rows yet.")
else:
    print("Portfolio performance file not found yet.")

print("\n--- Important Output Files ---")
for name, path in FILES.items():
    if path.exists():
        size = path.stat().st_size / 1024
        print(f"{name:<25} {str(path):<70} {size:>8.1f} KB")

print("\n--- Figures ---")
for fig in sorted(PATHS["outputs_figures"].glob("*.png")):
    print(fig.name)


FINAL RESULTS SUMMARY

Total performance rows: 5

--- Top Configurations by Sharpe Ratio ---


,window,top_m,score_type,model,method,Ann_Return,Ann_Vol,Sharpe,N_Months
3,70_30_split,10,ml_predicted_return,RF,TopN_EqualWeight,0.2650,0.3214,0.6221,5
0,70_30_split,10,ml_predicted_return,GB,TopN_EqualWeight,0.1303,0.3334,0.1957,5
1,70_30_split,10,ml_predicted_return,LASSO,TopN_EqualWeight,0.0622,0.3570,-0.0079,5
4,70_30_split,49,equal_weight,Equal,EqualWeight,-0.0101,0.2474,-0.3035,5
2,70_30_split,10,ml_predicted_return,Linear,TopN_EqualWeight,-0.1799,0.1654,-1.4810,5



--- Average Sharpe by Model ---


model
RF        0.6221
GB        0.1957
LASSO    -0.0079
Equal    -0.3035
Linear   -1.4810
Name: Sharpe, dtype: float64


--- Important Output Files ---
companies                 data\raw\nifty50_companies.csv      2.5 KB
raw_news                  data\raw\raw_news.csv   4744.7 KB
cleaned_news              data\processed\cleaned_news.csv   4915.8 KB
scored_news               data\processed\scored_news.csv   5221.3 KB
daily_scores              data\processed\daily_scores.csv    693.6 KB
stock_returns             data\processed\stock_returns.csv   2423.3 KB
model_dataset             data\processed\model_dataset.csv   3223.5 KB
model_performance         outputs\models\model_performance.csv      0.3 KB
portfolio_perf            outputs\tables\portfolio_performance.csv      0.5 KB
portfolio_returns         outputs\tables\portfolio_returns.csv      2.1 KB

--- Figures ---
article_count_by_company.png
cumulative_returns.png
model_ranking.png
